<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 (ข้อมูลของฉัน): ตัวแปรเรดาร์แบบ polar จากไฟล์ UF ดิบใน Google Drive
### Polar radar variables, the Doppler dilemma, beam geometry and data-quality checks — **your own KKN volume**

ดัดแปลงจาก [`Teach_HID_Doppler_KKN_summerStorm/notebooks/01_polar_radar_variables.ipynb`](https://github.com/nattaponm/Teach_HID_Doppler_KKN_summerStorm)
ให้ใช้ **ไฟล์เรดาร์ดิบของคุณเอง** (เช่น `KKN240@202405030800.uf.bz2`) จาก Google Drive แทนข้อมูลที่เตรียมไว้ใน repo

| | ต้นฉบับ | ฉบับนี้ |
|---|---|---|
| ข้อมูลเข้า | CfRadial ที่ผ่าน QC แล้วใน `data/polar_all/` | **ไฟล์ UF ดิบ** (`.uf`, `.uf.bz2`, `.uf.gz`) จาก Drive |
| การเตรียมข้อมูล | ทำไว้แล้วใน NB00 | ทำ **gentle QC ชุดเดียวกับ NB00** ในโน้ตบุ๊กนี้ แล้วบันทึกเป็นไฟล์รูปแบบเดียวกัน → ฟังก์ชันใน `utils/` ใช้ต่อได้ทันที |
| วัน/เวลา | 3 พ.ค. 2567 | อ่านจาก **ชื่อไฟล์** (UTC) — ใช้กับวันอื่นได้ |
| จุดรายงานความเสียหาย | แสดงเสมอ | แสดงเฉพาะเมื่อข้อมูลเป็นวันที่ 3 พ.ค. 2567 |
| จุดที่สนใจ (กรอบซูม, ray ตัวอย่าง) | นากลาง/หนองบัวลำภู | **หาแกนพายุที่แรงที่สุดให้อัตโนมัติ** หรือกำหนดเองใน CONFIG |
| sounding | 48431 วันที่ 3 พ.ค. 2567 | ใช้ของ repo ถ้าวันตรงกัน หรือใส่ไฟล์ของคุณเอง |
| แบบฝึกหัด 1.2–1.3 | 8 volume ช่วงพีค | volume ของคุณ (ไฟล์เป้าหมาย ± ช่วงเวลาที่กำหนด) |

### 🚀 วิธีใช้
1. แก้เซลล์ **🔧 CONFIG** (อย่างน้อย `MY_DRIVE_FOLDER` และ `TARGET_FILE`)
2. **Runtime → Run all** (ครั้งแรก ~3–4 นาที: ติดตั้งแพ็กเกจ + clone repo ต้นฉบับเพื่อใช้ `utils/` และแผนที่ฐาน)
3. ผลลัพธ์ (ไฟล์ polar หลัง QC และรูป PNG/PDF 300 dpi) บันทึกใน `OUT_DIR` บน Drive ของคุณ

> **เข้าถึงไฟล์ใน Drive ได้ 2 วิธี**
> - **A. Mount Drive (แนะนำ):** ถ้าโฟลเดอร์ `Radar_data_PHD` เป็นของคุณ ระบุ path ใน `MY_DRIVE_FOLDER` (ถ้าไม่แน่ใจ ปล่อย `None` แล้วโน้ตบุ๊กจะค้นหาให้)
>   ถ้าโฟลเดอร์ถูกแชร์มาจากคนอื่น ให้คลิกขวาที่โฟลเดอร์ใน Drive → **Organize → Add shortcut → My Drive** ก่อน
> - **B. ลิงก์แชร์ (สำรอง):** ใช้ `gdown` ดาวน์โหลดเฉพาะไฟล์ที่ต้องการจาก `DRIVE_FOLDER_URL` — โฟลเดอร์ต้องแชร์แบบ *Anyone with the link* และ gdown อ่านได้ไม่เกิน 50 ไฟล์ต่อโฟลเดอร์

---
## 🎯 วัตถุประสงค์การเรียนรู้
เมื่อจบ notebook นี้ นิสิตจะสามารถ
1. **อ่านไฟล์ UF ดิบของ KKN** (ที่บีบอัดแบบ bzip2 หรือ gzip) และทำ **gentle QC** ให้ได้ไฟล์ CfRadial ขนาดเล็ก
2. **อธิบายโครงสร้างข้อมูล polar volume** (sweep → ray → gate) และอ่านไฟล์ CfRadial ด้วย **Py-ART**
3. **อธิบายความหมายทางกายภาพ** ของ $Z_H$, $V_r$, $\sigma_v$, $Z_{DR}$, $\Phi_{DP}$ และ $\rho_{HV}$
4. **คำนวณ Nyquist velocity และระยะไกลสุดที่ไม่กำกวม** แล้วอธิบาย *Doppler dilemma*
5. **คำนวณความสูงของลำคลื่นเรดาร์** แล้วเทียบกับระดับ 0 °C และ −20 °C จาก sounding
6. **ตรวจคุณภาพข้อมูลเบื้องต้น** ได้แก่ qc_flag, ZDR bias, $\rho_{HV}$ ตามระยะ และการพับเฟสของ $\Phi_{DP}$

## 📖 1. แนวคิดและสมการ

### 1.1 ข้อมูลเรดาร์แบบ polar
เรดาร์หมุนรอบตัว (azimuth 0–360°) ที่มุมยก (elevation) คงที่ 1 รอบ = **sweep** หรือ PPI
แต่ละทิศทางคือ **ray** และแต่ละ ray แบ่งตามระยะเป็น **gate** (KKN: 480 gate × 500 m = 240 km)
KKN ในเคสนี้มี **4 มุมยก (0.48°, 1.48°, 2.39°, 3.39°)** ทุก 15 นาที รวมเป็น 1 **volume**

### 1.2 ตัวแปรหลัก
| สัญลักษณ์ | ชื่อ | หน่วย | บอกอะไร |
|---|---|---|---|
| $Z_H$ | reflectivity (การสะท้อนกลับ) | dBZ | ขนาดและจำนวนเม็ดน้ำ/น้ำแข็ง ($\propto \sum D^6$) |
| $V_r$ | radial velocity (ความเร็วตามแนวรัศมี) | m s⁻¹ | ลมที่วิ่ง **เข้าหา (−)** หรือ **ออกจาก (+)** เรดาร์ |
| $\sigma_v$ | spectrum width (ความกว้างสเปกตรัม) | m s⁻¹ | ความแปรปรวนของความเร็วในปริมาตรวัด → ความปั่นป่วน (NB3) |
| $Z_{DR}$ | differential reflectivity | dB | รูปร่างเม็ด: ฝนเม็ดใหญ่แบน → ค่าบวกมาก, ลูกเห็บกลิ้ง → ใกล้ 0 |
| $\Phi_{DP}$ | differential phase | ° | เฟสสะสมจากน้ำเหลว → ใช้หา $K_{DP}$ (NB4) |
| $\rho_{HV}$ | co-polar correlation coefficient | – | ความ "เหมือนกัน" ของเม็ดในปริมาตร: ฝนล้วน > 0.97, ปนลูกเห็บ/ชั้นละลาย 0.85–0.95, non-met < 0.8 |

$$ Z_{DR} = 10\log_{10}\frac{Z_H}{Z_V},\qquad K_{DP} = \frac{1}{2}\frac{d\Phi_{DP}}{dr} $$

### 1.3 หลักการ Doppler และ *Doppler dilemma*
เรดาร์วัดการเลื่อนเฟสระหว่างพัลส์ ได้ความถี่ Doppler $f_D = 2V_r/\lambda$ แต่วัดได้ไม่เกินครึ่งหนึ่งของ PRF (Nyquist)
$$ V_{ny} = \frac{\lambda\,\mathrm{PRF}}{4},\qquad r_{max} = \frac{c}{2\,\mathrm{PRF}}
\quad\Longrightarrow\quad V_{ny}\,r_{max} = \frac{c\,\lambda}{8} $$
ผลคูณคงที่ → ถ้าอยากวัด **ไกล** ต้องใช้ PRF ต่ำ → $V_{ny}$ **ต่ำ** → ลมแรงเกิน $\pm V_{ny}$ จะ **"พับ" (aliasing)** กลับมาเป็นค่าผิด
$$ V_{obs} = V_{true} - 2nV_{ny},\qquad n \in \{\dots,-1,0,1,\dots\} $$

### 1.4 ความสูงของลำคลื่น (แบบจำลองโลกรัศมี 4/3)
$$ h = \sqrt{r^2 + (k_e a)^2 + 2 r k_e a \sin\theta} - k_e a + h_0,\qquad k_e = 4/3,\ a = 6371\ \mathrm{km} $$
ความกว้างลำคลื่น $\approx r\,\Delta\theta$ (beamwidth 1°) → ที่ 120 km ลำคลื่นกว้าง ~2 km
ทำให้ "สิ่งที่เรดาร์เห็น" ไกล ๆ เป็นค่าเฉลี่ยของปริมาตรใหญ่ และอาจอยู่ **เหนือระดับเยือกแข็ง** ทั้งหมด

## ⚙️ 2. เตรียมสภาพแวดล้อมและข้อมูล
### 2.0 ติดตั้งแพ็กเกจ, clone repo ต้นฉบับ (เพื่อใช้ `utils/` และแผนที่ฐาน) และ mount Google Drive

In [ ]:
#@title ⚙️ Setup: ติดตั้งแพ็กเกจ + clone repo + mount Drive (รันก่อนเสมอ ~2–3 นาที)
import os, sys, warnings
warnings.filterwarnings("ignore")

REPO = "Teach_HID_Doppler_KKN_summerStorm"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q arm_pyart cartopy netCDF4 geopandas metpy cmweather rasterio "gdown>=5"
    if not os.path.exists(REPO):
        !git clone -q --depth 1 https://github.com/nattaponm/{REPO}.git
    from google.colab import drive
    drive.mount("/content/drive")

for ROOT in [REPO, ".", "..", os.path.join("..", REPO)]:
    if os.path.exists(os.path.join(ROOT, "utils", "polar_tools.py")):
        break
else:
    raise FileNotFoundError(f"ไม่พบ {REPO}/utils — รันเซลล์นี้ใน Colab หรือ git clone repo ไว้ข้างโน้ตบุ๊ก")
ROOT = os.path.abspath(ROOT)
sys.path.insert(0, ROOT)
DATA = os.path.join(ROOT, "data")          # แผนที่ฐาน, sounding และจุดรายงานของ repo ต้นฉบับ
print("✅ utils/ จาก:", ROOT)

### 🔧 2.1 CONFIG — แก้เฉพาะเซลล์นี้

In [ ]:
#@title 🔧 CONFIG
from pathlib import Path

# ---------- ไฟล์เรดาร์ของคุณ ----------
TARGET_FILE = "KKN240@202405030800.uf.bz2"        # ไฟล์หลักที่จะวิเคราะห์ (เวลาในชื่อไฟล์ = UTC)

# วิธี A: path ของโฟลเดอร์ใน Drive หลัง mount (ค้นหาไฟล์ในโฟลเดอร์ย่อยให้อัตโนมัติ)
#         None = ค้นหาโฟลเดอร์ชื่อ DRIVE_FOLDER_NAME ใน MyDrive ให้เอง
MY_DRIVE_FOLDER = None                              # เช่น "/content/drive/MyDrive/Radar_data_PHD"
DRIVE_FOLDER_NAME = "Radar_data_PHD"
# วิธี B (สำรอง): ลิงก์แชร์ของโฟลเดอร์ → ดาวน์โหลดด้วย gdown
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/11G1JOhI1MMqoeCSAmifyL2RM6HCxjzxj"

# volume ข้างเคียงสำหรับแบบฝึกหัด 1.2–1.3 และกราฟ Nyquist (นาที ก่อน/หลังไฟล์หลัก; 0 = ใช้ไฟล์เดียว)
NEIGHBOUR_WINDOW_MIN = 30

# ---------- sounding ----------
# None = ใช้ไฟล์ใน repo ถ้าวันตรงกัน (มีเฉพาะ 48431 วันที่ 3 พ.ค. 2567 00Z)
# หรือใส่ path ของ CSV ของคุณ (คอลัมน์: pressure, height, temperature, dewpoint, direction, speed — แบบ U. Wyoming)
SOUNDING_CSV = None
DEFAULT_ISOTHERMS_M = {0: 5000.0, -10: 6700.0, -20: 8400.0}   # ใช้เมื่อไม่มี sounding (ค่าประมาณฤดูร้อนอีสาน)

# ---------- พื้นที่ที่สนใจ ----------
FOCUS_LONLAT = None          # None = หาแกนพายุที่แรงที่สุดให้อัตโนมัติ, หรือ (lon, lat) เช่น (102.35, 17.21)
ZOOM_HALF_KM = 90            # ครึ่งความกว้างกรอบซูมของแผนที่ (km); None = แสดงทั้งรัศมี 240 km

# ---------- ผลลัพธ์ ----------
OUT_DIR = "/content/drive/MyDrive/Radar_data_PHD_output" if IN_COLAB else "output_mydata"

# ---------- gentle QC (ค่าเดียวกับ NB00 ของ repo) ----------
ZH_ECHO_FLOOR_DBZ = 0.0
NONMET_RHOHV_MAX = 0.70
NONMET_ZH_MAX_DBZ = 25.0
SPECKLE_MIN_GATES = 12
FLARE_MIN_LEN_KM = 120.0
FLARE_ZH_MIN_DBZ = 5.0
FLARE_RHOHV_MEAN_MAX = 0.75
FLARE_VELVALID_MAX = 0.30
FLARE_PROTECT_DBZ = 45.0
FLARE_PROTECT_FRAC = 0.03
SW_MAX = 12.0
QC_SD_PHIDP_W = 5
QC_SD_PHIDP_MAX = 20.0

# ใช้ทดสอบนอก Colab: ตั้งตัวแปรสภาพแวดล้อม MY_DRIVE_FOLDER / OUT_DIR ได้
MY_DRIVE_FOLDER = os.environ.get("MY_DRIVE_FOLDER", MY_DRIVE_FOLDER)
OUT_DIR = os.environ.get("OUT_DIR", OUT_DIR)
POLAR_DIR = Path(OUT_DIR) / "polar"                 # ไฟล์ polar หลัง QC (รูปแบบเดียวกับ repo)
FIG_DIR = Path(OUT_DIR) / "figures"
RAW_CACHE = Path("raw_cache")                       # ที่เก็บไฟล์ที่ดาวน์โหลดด้วย gdown (วิธี B)
for d in (POLAR_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("ผลลัพธ์จะอยู่ที่:", Path(OUT_DIR).resolve())

### 2.2 ฟังก์ชันอ่านไฟล์ UF และ gentle QC
ชุดเดียวกับ `00_prepare_polar_for_doppler_pytda_hid.ipynb` ของ repo:
- ไฟล์ของกรมอุตุฯ ที่ชื่อลงท้าย `.uf.bz2` **มักเป็น gzip จริง** → ตรวจชนิดการบีบอัดจาก *magic bytes* แทนนามสกุล
- **gentle QC:** เก็บ $Z_H$ ≥ 0 dBZ, ตัด flare ray (สัญญาณรบกวน/ดวงอาทิตย์) แต่ **ไม่ตัด ray ที่มีแกนพายุ**, ตัด non-met ($\rho_{HV}$ < 0.7 และ Z < 25 dBZ), ตัด speckle < 12 gate
- บันทึกเป็น CfRadial แบบ int16 + zlib ที่มี `reflectivity_raw`, `*_qc` และ `qc_flag` เหมือนไฟล์ใน repo ทุกประการ

In [ ]:
#@title 🧰 ฟังก์ชัน: ค้นหาไฟล์ใน Drive, อ่าน UF, gentle QC, เขียน CfRadial
import re, bz2, gzip, shutil, tempfile, datetime
import numpy as np
import pandas as pd
from scipy.ndimage import binary_dilation, uniform_filter1d
from scipy.ndimage import label as cc_label
import pyart

PREP_VERSION = "kkn-prep-1.0.0 (MYDATA)"
RAW_RE = re.compile(r"^([A-Za-z]+)\d*@(\d{8})(\d{4})\.uf(\.bz2|\.gz)?$")


def parse_raw_name(name):
    """'KKN240@202405030800.uf.bz2' → ('KKN', datetime UTC) หรือ None"""
    m = RAW_RE.match(Path(name).name)
    if not m:
        return None
    return m.group(1).upper(), datetime.datetime.strptime(m.group(2) + m.group(3), "%Y%m%d%H%M")


# ---------------- หาไฟล์: วิธี A (mount) ----------------
def find_drive_folder():
    if MY_DRIVE_FOLDER:
        p = Path(MY_DRIVE_FOLDER)
        if not p.exists():
            raise FileNotFoundError(f"ไม่พบโฟลเดอร์ {p} — ตรวจ path ใน CONFIG")
        return p
    base = Path("/content/drive/MyDrive")
    if not base.exists():
        return None
    for depth in range(0, 4):                      # ค้นหาไม่เกิน 4 ชั้น (เร็วกว่า rglob ทั้ง Drive)
        hits = list(base.glob("/".join(["*"] * depth + [DRIVE_FOLDER_NAME])))
        if hits:
            return hits[0]
    return None


def files_from_mount(folder, target, window_min):
    _, t0 = parse_raw_name(target)
    found = {}
    for f in folder.rglob("*.uf*"):
        info = parse_raw_name(f.name)
        if info and abs((info[1] - t0).total_seconds()) <= window_min * 60:
            found.setdefault(f.name, f)
    return found


# ---------------- หาไฟล์: วิธี B (gdown) ----------------
def files_from_gdown(url, target, window_min):
    import gdown
    _, t0 = parse_raw_name(target)
    listing = gdown.download_folder(url=url, skip_download=True, quiet=True)
    RAW_CACHE.mkdir(exist_ok=True)
    found = {}
    for item in listing:
        name = Path(item.path).name
        info = parse_raw_name(name)
        if info and abs((info[1] - t0).total_seconds()) <= window_min * 60:
            out = RAW_CACHE / name
            if not out.exists():
                gdown.download(id=item.id, output=str(out), quiet=True)
            found[name] = out
    return found


def locate_raw_files():
    """คืน dict {ชื่อไฟล์: path} ของไฟล์หลัก + ไฟล์ข้างเคียง"""
    if parse_raw_name(TARGET_FILE) is None:
        raise ValueError(f"ชื่อไฟล์ไม่ตรงรูปแบบ RADAR###@YYYYMMDDhhmm.uf[.bz2|.gz]: {TARGET_FILE}")
    folder = find_drive_folder()
    if folder is not None:
        print("วิธี A: ค้นหาใน", folder)
        found = files_from_mount(folder, TARGET_FILE, NEIGHBOUR_WINDOW_MIN)
        if TARGET_FILE in found:
            return found
        print(f"  ไม่พบ {TARGET_FILE} ในโฟลเดอร์นี้ → ลองวิธี B")
    print("วิธี B: ดาวน์โหลดจากลิงก์แชร์ด้วย gdown")
    try:
        found = files_from_gdown(DRIVE_FOLDER_URL, TARGET_FILE, NEIGHBOUR_WINDOW_MIN)
    except Exception as e:
        raise RuntimeError(
            "ดาวน์โหลดด้วย gdown ไม่สำเร็จ: " + repr(e)[:300] +
            "\n→ ตรวจว่าแชร์โฟลเดอร์แบบ 'Anyone with the link' หรือใช้วิธี A: Add shortcut ไปยัง My Drive "
            "แล้วใส่ path ใน MY_DRIVE_FOLDER") from e
    if TARGET_FILE not in found:
        raise FileNotFoundError(f"ไม่พบ {TARGET_FILE} ในโฟลเดอร์ Drive")
    return found


# ---------------- อ่าน UF ----------------
def read_radar_magic(path, verbose=True):
    """อ่าน UF โดยดูชนิดการบีบอัดจาก magic bytes (ไฟล์ .uf.bz2 ของกรมอุตุฯ มักเป็น gzip)"""
    path = Path(path)
    with open(path, "rb") as f:
        magic = f.read(4)
    if magic[:3] == b"BZh":
        opener, kind = bz2.open, "bzip2"
    elif magic[:2] == b"\x1f\x8b":
        opener, kind = gzip.open, "gzip"
    else:
        opener, kind = None, "uncompressed"
    tmp = tempfile.NamedTemporaryFile(suffix=".uf", delete=False)
    tmp.close()
    try:
        if opener is None:
            read_path = str(path)
        else:
            with opener(path, "rb") as fin, open(tmp.name, "wb") as fout:
                shutil.copyfileobj(fin, fout)
            read_path = tmp.name
        try:
            radar = pyart.io.read(read_path)
        except Exception:
            radar = pyart.io.read_uf(read_path)
    finally:
        if os.path.exists(tmp.name):
            os.unlink(tmp.name)
    if verbose:
        print(f"อ่าน {path.name} [{kind}]")
    return radar


def pick_field(radar, candidates):
    return next((c for c in candidates if c in radar.fields), None)


def farr(radar, field):
    if field is None or field not in radar.fields:
        return None
    return np.ma.filled(radar.fields[field]["data"].astype(np.float32), np.nan)


def detect_fields(radar):
    f = {"REFL": pick_field(radar, ["corrected_reflectivity", "reflectivity"]),
         "VEL": pick_field(radar, ["velocity", "corrected_velocity"]),
         "SW": pick_field(radar, ["spectrum_width"]),
         "ZDR": pick_field(radar, ["corrected_differential_reflectivity", "differential_reflectivity"]),
         "PHIDP": pick_field(radar, ["differential_phase", "uncorrected_differential_phase"]),
         "RHO": pick_field(radar, ["cross_correlation_ratio"])}
    if f["REFL"] is None:
        raise ValueError(f"ไม่พบ reflectivity ในไฟล์ (fields ที่มี: {list(radar.fields)})")
    return f


def get_nyquist(radar):
    try:
        v = np.asarray(radar.instrument_parameters["nyquist_velocity"]["data"], float)
        v = v[np.isfinite(v) & (v > 0)]
        return float(np.nanmedian(v)) if v.size else np.nan
    except Exception:
        return np.nan


def sweep_index_1d(radar):
    swp = np.zeros(radar.nrays, dtype=np.int16)
    for k in range(radar.nsweeps):
        swp[radar.sweep_start_ray_index["data"][k]:radar.sweep_end_ray_index["data"][k] + 1] = k
    return swp


# ---------------- gentle QC ----------------
QC_BITS_FULL = {"no_echo_or_below_floor": 1, "flare_ray": 2, "nonmet_low_rhohv_weak_z": 4, "speckle": 8,
                "spectrum_width_invalid": 16, "velocity_invalid": 32, "phidp_texture_high": 64}


def find_flare_rays(radar, zh, rho, vel_finite):
    dr_km = float(np.nanmedian(np.diff(radar.range["data"])) / 1000.0)
    min_gate_count = int(FLARE_MIN_LEN_KM / max(dr_km, 0.1))
    valid = np.isfinite(zh) & (zh > FLARE_ZH_MIN_DBZ)
    spike = np.zeros(radar.nrays, dtype=bool)
    for i in range(radar.nrays):
        v = valid[i]
        if v.sum() < min_gate_count:
            continue
        if np.mean(zh[i][v] >= FLARE_PROTECT_DBZ) >= FLARE_PROTECT_FRAC:
            continue                                   # ray ที่มีแกนพายุจริง → ไม่ตัด
        rho_ok = True if rho is None else np.nanmean(rho[i][v]) <= FLARE_RHOHV_MEAN_MAX
        vel_ok = True if vel_finite is None else np.mean(vel_finite[i][v]) <= FLARE_VELVALID_MAX
        spike[i] = rho_ok and vel_ok
    return spike, binary_dilation(spike, iterations=1)


def remove_speckles_polar(mask_keep, radar, min_gates=12, pad=8):
    keep = mask_keep.copy()
    removed = np.zeros_like(keep, dtype=bool)
    swp = sweep_index_1d(radar)
    st = np.ones((3, 3), dtype=np.int8)
    for k in range(radar.nsweeps):
        ridx = np.where(swp == k)[0]
        if len(ridx) == 0:
            continue
        sub = keep[ridx, :].copy()
        if len(ridx) > 2 * pad:                       # ต่อ azimuth 360° ↔ 0°
            lab_p, _ = cc_label(np.vstack([sub[-pad:], sub, sub[:pad]]), structure=st)
            sizes = np.bincount(lab_p.ravel())
            lab = lab_p[pad:pad + len(ridx), :]
        else:
            lab, _ = cc_label(sub, structure=st)
            sizes = np.bincount(lab.ravel())
        small = np.isin(lab, np.where(sizes < min_gates)[0]) & (lab > 0)
        sub[small] = False
        keep[ridx, :] = sub
        removed[ridx, :] = small
    return keep, removed


def circular_sd_deg_range(data_deg, window=5):
    x = np.asarray(data_deg, dtype=float)
    valid = np.isfinite(x)
    a = np.deg2rad(np.where(valid, x, 0.0))
    cnt = uniform_filter1d(valid.astype(float), window, axis=1, mode="nearest") * window
    s = uniform_filter1d(np.where(valid, np.sin(a), 0), window, axis=1, mode="nearest") * window
    c = uniform_filter1d(np.where(valid, np.cos(a), 0), window, axis=1, mode="nearest") * window
    with np.errstate(invalid="ignore", divide="ignore"):
        R = np.clip(np.hypot(s / cnt, c / cnt), 1e-6, 1.0)
        sd = np.rad2deg(np.sqrt(-2.0 * np.log(R)))
    sd[cnt < 3] = np.nan
    return sd.astype(np.float32)


def gentle_base_qc(radar, fields, verbose=True):
    zh, rho = farr(radar, fields["REFL"]), farr(radar, fields["RHO"])
    vel, sw, phidp = farr(radar, fields["VEL"]), farr(radar, fields["SW"]), farr(radar, fields["PHIDP"])
    vel_finite = None if vel is None else np.isfinite(vel)
    echo = np.isfinite(zh) & (zh >= ZH_ECHO_FLOOR_DBZ)
    _, flare_dil = find_flare_rays(radar, zh, rho, vel_finite)
    flare = np.zeros_like(echo)
    flare[flare_dil, :] = True
    nonmet = (echo & ~flare & np.isfinite(rho) & (rho < NONMET_RHOHV_MAX) & (zh < NONMET_ZH_MAX_DBZ)
              if rho is not None else np.zeros_like(echo))
    keep, speckle = remove_speckles_polar(echo & ~flare & ~nonmet, radar, min_gates=SPECKLE_MIN_GATES)
    out = {"zh_raw": zh, "keep": keep, "reflectivity_qc": np.where(keep, zh, np.nan).astype(np.float32)}
    flag = np.zeros(zh.shape, dtype=np.uint8)
    flag[~echo] |= 1
    flag[flare] |= 2
    flag[nonmet] |= 4
    flag[speckle] |= 8
    if vel is not None:
        ok = keep & np.isfinite(vel)
        out["velocity_qc"] = np.where(ok, vel, np.nan).astype(np.float32)
        flag[keep & ~ok] |= 32
    if sw is not None:
        ok = keep & np.isfinite(sw) & (sw >= 0.0) & (sw <= SW_MAX)
        out["spectrum_width_qc"] = np.where(ok, sw, np.nan).astype(np.float32)
        flag[keep & ~ok] |= 16
    if phidp is not None:
        ok = np.isfinite(phidp) & (circular_sd_deg_range(phidp, QC_SD_PHIDP_W) < QC_SD_PHIDP_MAX)
        flag[keep & ~ok] |= 64
    out["qc_flag"] = flag
    n_raw = int(np.isfinite(zh).sum())
    out["counts"] = {"raw_finite": n_raw, "echo_floor": int(echo.sum()), "flare_rays": int(flare_dil.sum()),
                     "nonmet_removed": int(nonmet.sum()), "speckle_removed": int(speckle.sum()),
                     "kept": int(keep.sum()), "kept_pct_of_raw": round(100.0 * keep.sum() / max(n_raw, 1), 2)}
    if verbose:
        for k, v in out["counts"].items():
            print(f"  {k:<18}: {v}")
    return out


# ---------------- เขียน CfRadial ขนาดเล็ก (รูปแบบเดียวกับ repo) ----------------
POLAR_SPECS = {
    "reflectivity_raw":             ("REFL_RAW", "reflectivity", 0.1),
    "reflectivity_qc":              ("reflectivity_qc", "reflectivity", 0.1),
    "velocity_qc":                  ("velocity_qc", "velocity", 0.1),
    "spectrum_width_qc":            ("spectrum_width_qc", "spectrum_width", 0.1),
    "differential_reflectivity_qc": ("ZDR_QC", "differential_reflectivity", 0.01),
    "differential_phase_qc":        ("PHIDP_QC", "differential_phase", 0.1),
    "cross_correlation_ratio_qc":   ("RHO_QC", "cross_correlation_ratio", 0.001),
}


def packed_field(data, base_name, long_name, scale, chunks):
    f = pyart.config.get_metadata(base_name)
    f.pop("valid_min", None); f.pop("valid_max", None)
    f.update({"data": np.ma.masked_invalid(np.asarray(data, np.float32)), "long_name": long_name,
              "_Write_as_dtype": "int16", "scale_factor": np.float32(scale), "add_offset": np.float32(0.0),
              "_FillValue": np.int16(-32768), "_DeflateLevel": 4, "_Shuffle": True, "_ChunkSizes": chunks})
    return f


def write_lean_polar(radar, fields, qc, out_path, isotherms, sounding_name):
    keep = qc["keep"]
    src = {"REFL_RAW": qc["zh_raw"], "reflectivity_qc": qc["reflectivity_qc"],
           "velocity_qc": qc.get("velocity_qc"), "spectrum_width_qc": qc.get("spectrum_width_qc"),
           "ZDR_QC": None if fields["ZDR"] is None else np.where(keep, farr(radar, fields["ZDR"]), np.nan),
           "PHIDP_QC": None if fields["PHIDP"] is None else np.where(keep, farr(radar, fields["PHIDP"]), np.nan),
           "RHO_QC": None if fields["RHO"] is None else np.where(keep, farr(radar, fields["RHO"]), np.nan)}
    chunks = (int(np.max(radar.rays_per_sweep["data"])), radar.ngates)
    written = []
    for name, (key, base, scale) in POLAR_SPECS.items():
        if src.get(key) is None:
            print(f"  ⚠️ ไม่มี {name} (ไม่มีตัวแปรต้นทางในไฟล์ UF)")
            continue
        radar.add_field(name, packed_field(src[key], base, f"{name} (gentle base-QC)", scale, chunks),
                        replace_existing=True)
        written.append(name)
    radar.add_field("qc_flag", {"data": qc["qc_flag"].astype(np.uint8), "units": "1",
                                "long_name": "base-QC flag (bitmask)",
                                "flag_masks": np.array(list(QC_BITS_FULL.values()), np.uint8),
                                "flag_meanings": " ".join(QC_BITS_FULL),
                                "_DeflateLevel": 4, "_Shuffle": True, "_ChunkSizes": chunks},
                    replace_existing=True)
    written.append("qc_flag")
    radar.metadata.update({
        "title": "KKN lean polar volume (gentle base-QC)", "processing_code": PREP_VERSION,
        "velocity_note": f"velocity is NOT dealiased; Nyquist = {get_nyquist(radar):.2f} m/s",
        "sounding_file": sounding_name, **{k: f"{v:.0f}" for k, v in isotherms.items()},
        "history": f"created {datetime.datetime.utcnow():%Y-%m-%dT%H:%M:%SZ}"})
    tmp = str(out_path) + ".tmp"
    pyart.io.write_cfradial(tmp, radar, include_fields=written)
    os.replace(tmp, out_path)
    return written


print("✅ ฟังก์ชันพร้อมใช้งาน")

### 2.3 ค้นหาไฟล์ → อ่าน UF → QC → บันทึกไฟล์ polar

In [ ]:
RADAR_NAME, T_TARGET = parse_raw_name(TARGET_FILE)
CASE_DATE = T_TARGET.strftime("%Y%m%d")

# ---- sounding: ไฟล์ของผู้ใช้ > ไฟล์ใน repo ที่วันตรงกัน > ค่าตั้งต้น ----
snd_path = SOUNDING_CSV
if snd_path is None:
    cand = sorted(Path(DATA, "sounding").glob(f"*_{T_TARGET:%Y-%m-%d}_*.csv"))
    snd_path = str(cand[0]) if cand else None
if snd_path:
    _s = pd.read_csv(snd_path).dropna(subset=["height", "temperature"]).sort_values("height")
    h_, t_ = _s["height"].values, _s["temperature"].values
    ISOTHERMS = {}
    for lev in (0, -10, -20):
        i = np.where((t_[:-1] >= lev) & (t_[1:] < lev))[0]
        if i.size:
            i = i[0]
            ISOTHERMS[f"height_{lev}C_m_msl"] = float(h_[i] + (lev - t_[i]) * (h_[i + 1] - h_[i]) / (t_[i + 1] - t_[i]))
    print("sounding:", Path(snd_path).name)
else:
    ISOTHERMS = {f"height_{k}C_m_msl": v for k, v in DEFAULT_ISOTHERMS_M.items()}
    print("⚠️ ไม่มี sounding ของวันนี้ → ใช้ระดับอุณหภูมิค่าตั้งต้น", DEFAULT_ISOTHERMS_M,
          "(ใส่ SOUNDING_CSV ใน CONFIG เพื่อความถูกต้อง)")

# ---- หาไฟล์ ----
raw = locate_raw_files()
raw = dict(sorted(raw.items(), key=lambda kv: parse_raw_name(kv[0])[1]))
print(f"พบ {len(raw)} ไฟล์ (ไฟล์หลัก ± {NEIGHBOUR_WINDOW_MIN} นาที):", list(raw))

# ---- ประมวลผลทีละไฟล์ (ไฟล์หลักแสดงรายละเอียด QC) ----
rows = []
for name, path in raw.items():
    _, t = parse_raw_name(name)
    out_nc = POLAR_DIR / f"{RADAR_NAME}_{t:%Y%m%d_%H%M}_polar.nc"
    verbose = (name == TARGET_FILE)
    try:
        rdr = read_radar_magic(path, verbose=verbose)
        flds = detect_fields(rdr)
        if verbose:
            print(f"  ตำแหน่งเรดาร์: {float(rdr.latitude['data'][0]):.4f} N, {float(rdr.longitude['data'][0]):.4f} E, "
                  f"{float(rdr.altitude['data'][0]):.0f} m | sweeps {rdr.nsweeps}, rays {rdr.nrays}, gates {rdr.ngates}")
            print(f"  มุมยก: {np.round(rdr.fixed_angle['data'], 2).tolist()} | Nyquist {get_nyquist(rdr):.2f} m/s")
            print(f"  ตัวแปรที่ใช้: {flds}")
            print("  ผล gentle QC:")
        q = gentle_base_qc(rdr, flds, verbose=verbose)
        write_lean_polar(rdr, flds, q, out_nc, ISOTHERMS, Path(snd_path).name if snd_path else "none (default isotherms)")
        rows.append({"file": name, "time_utc": t, "nyquist": get_nyquist(rdr), "status": "ok",
                     "polar_nc": str(out_nc), "size_MB": round(out_nc.stat().st_size / 1e6, 2), **q["counts"]})
    except Exception as e:
        rows.append({"file": name, "time_utc": t, "status": f"failed: {e!r}"[:200]})
        if verbose:
            raise
VOLSTATS = pd.DataFrame(rows)
VOLSTATS.to_csv(Path(OUT_DIR) / f"{RADAR_NAME}_{CASE_DATE}_volume_stats.csv", index=False)
VOLSTATS[["file", "status", "nyquist", "kept_pct_of_raw", "size_MB"]]

In [ ]:
import matplotlib.pyplot as plt
from utils import polar_tools as pt
from utils import mapplot as mp
from utils import kkn_tools as kt
mp.set_style()
mp.set_data_dir(DATA)
mp.FIG_DIR = str(FIG_DIR)

VOLS = [r for r in VOLSTATS.loc[VOLSTATS["status"] == "ok", "polar_nc"]]     # volume ทั้งหมดที่ประมวลผลได้
target_nc = str(POLAR_DIR / f"{RADAR_NAME}_{T_TARGET:%Y%m%d_%H%M}_polar.nc")
radar = pt.read_volume(target_nc)
meta = pt.polar_meta(radar)                         # ระบบพิกัด AEQD สำหรับแผนที่

snd = pt.load_profile(snd_path) if snd_path else None
H = ({f"{t} °C": pt.isotherm_height(snd, t) for t in (0, -10, -20)} if snd is not None
     else {f"{t} °C": v for t, v in DEFAULT_ISOTHERMS_M.items()})

# จุดรายงานความเสียหายของ repo ใช้ได้เฉพาะวันที่ 3 พ.ค. 2567
reports = kt.load_reports(DATA, meta) if CASE_DATE == "20240503" else None

# จุดที่สนใจ: แกนพายุที่แรงที่สุด (Z เฉลี่ย 5×5 gate สูงสุด) ในมุมยก 1.48° หรือที่กำหนดใน CONFIG
if FOCUS_LONLAT is None:
    from scipy.ndimage import uniform_filter
    s_ = 1 if radar.nsweeps > 1 else 0
    zf = uniform_filter(np.nan_to_num(pt.field_data(radar, "reflectivity_qc", s_), nan=0.0), 5)
    ir, ig = np.unravel_index(np.argmax(zf), zf.shape)
    gx, gy, _ = radar.get_gate_x_y_z(s_)
    FOCUS = kt.xy_to_lonlat(meta, gx[ir, ig], gy[ir, ig])
    FOCUS_NAME = f"Strongest core ({zf[ir, ig]:.0f} dBZ)"
else:
    FOCUS, FOCUS_NAME = tuple(FOCUS_LONLAT), "Focus point"
FOCUS = (float(np.ravel(FOCUS[0])[0]), float(np.ravel(FOCUS[1])[0]))
mp.POINTS = [(FOCUS_NAME.split(" (")[0], FOCUS[0], FOCUS[1], "*")]
ZOOM = kt.box_around(meta, FOCUS[0], FOCUS[1], ZOOM_HALF_KM) if ZOOM_HALF_KM else None
TLAB = mp.tlabel(pt.volume_time(radar))

print(f"เวลา: {CASE_DATE} {TLAB} | จำนวน volume: {len(VOLS)}")
print(f"จุดที่สนใจ: {FOCUS_NAME} ที่ {FOCUS[0]:.3f} E, {FOCUS[1]:.3f} N")
print("ระดับอุณหภูมิ:", {k: f"{v:.0f} m" for k, v in H.items()})
print("จุดรายงานความเสียหาย:", "แสดง (เคส 3 พ.ค. 2567)" if reports is not None else "ไม่มีสำหรับวันนี้")

## 🔍 3. โครงสร้างของ polar volume

In [ ]:
print(f"sweeps = {radar.nsweeps}, rays ทั้งหมด = {radar.nrays}, gates/ray = {radar.ngates}")
print("มุมยก (°):", np.round(radar.fixed_angle["data"], 2))
print("rays ต่อ sweep:", [radar.get_slice(s).stop - radar.get_slice(s).start for s in range(radar.nsweeps)])
print(f"ระยะ gate แรก–สุดท้าย: {radar.range['data'][0]/1000:.2f}–{radar.range['data'][-1]/1000:.1f} km "
      f"(ทุก {np.diff(radar.range['data'])[0]:.0f} m)")
print(f"Nyquist velocity = {pt.nyquist(radar):.2f} m/s | ความถี่ = {pt.RADAR_FREQ_HZ/1e9:.1f} GHz (λ = {pt.WAVELENGTH_M*100:.2f} cm)")
pd.DataFrame({f: {"units": radar.fields[f].get("units", ""),
                  "valid_gates": int(np.ma.count(radar.fields[f]["data"])),
                  "min": float(np.ma.min(radar.fields[f]["data"])),
                  "max": float(np.ma.max(radar.fields[f]["data"]))} for f in radar.fields}).T

`reflectivity_raw` คือ Z ก่อน QC ส่วน field ที่ลงท้าย `_qc` ผ่าน **gentle QC** ในหัวข้อ 2.3 แล้ว
**ความเร็วยังไม่ได้แก้การพับ (aliased)** — จะทำใน NB2

> 💡 ถ้าจำนวนมุมยก, rays ต่อ sweep หรือ Nyquist **ต่างจากเคส 3 พ.ค. 2567** (4 มุมยก, ~511 ray, 8.38 m/s) แสดงว่า
> ยุทธวิธีการสแกน (scan strategy) ของวันนั้นต่างไป ให้จดไว้เพราะมีผลกับ NB2–NB5

## 🗺️ 4. แผนที่ 6 ตัวแปร ที่มุมยกต่ำสุด
ดาว (★) = จุดที่สนใจ (แกนพายุที่แรงที่สุด หรือ `FOCUS_LONLAT`), ◆/✕ = จุดรายงานความเสียหาย (เฉพาะเคส 3 พ.ค. 2567)

In [ ]:
panels = [("reflectivity_qc", "Z", "$Z_H$"), ("velocity_qc", "VA", "$V_r$ (aliased)"),
          ("spectrum_width_qc", "W", "$\\sigma_v$"), ("differential_reflectivity_qc", "ZDR", "$Z_{DR}$ (raw)"),
          ("differential_phase_qc", "PHI", "$\\Phi_{DP}$ (raw)"), ("cross_correlation_ratio_qc", "RHO", "$\\rho_{HV}$")]
panels = [p for p in panels if p[0] in radar.fields]
ncol = 3
nrow = int(np.ceil(len(panels) / ncol))
fig, axs = mp.map_panels(nrow, ncol, meta, panel_width=2.9, zoom=ZOOM)
for k, (ax, (f, kind, ttl)) in enumerate(zip(axs, panels)):
    im = pt.ppi_map(ax, radar, f, 0, meta, kind=kind, zoom=ZOOM, title=ttl, north=(k == ncol - 1),
                    gridline_labels=(("left",) if k % ncol == 0 else ()) + (("bottom",) if k >= len(panels) - ncol else ()))
    if reports is not None:
        mp.add_reports(ax, reports, legend=(k == 0))
    mp.add_colorbar(fig, im, ax, shrink=0.9, extend="both")
for ax in axs[len(panels):]:
    ax.set_visible(False)
mp.panel_labels(axs[:len(panels)])
fig.suptitle(f"{RADAR_NAME} {radar.fixed_angle['data'][0]:.2f}° PPI, {T_TARGET:%d %b %Y} {TLAB}", fontsize=10)
mp.save_figure(fig, f"nb1_six_variables_{CASE_DATE}_{T_TARGET:%H%M}")
plt.show()

**อ่านแผนที่ (ตรวจกับข้อมูลของคุณ):**
- (a) แกนพายุ $Z_H$ ≥ 45 dBZ อยู่ที่ไหน? ตรงกับดาว (★) หรือไม่
- (b) สีน้ำเงิน = ลมเข้าหาเรดาร์, แดง = ออกจากเรดาร์ — ถ้าเห็น **สีกระโดดจากแดงเข้มเป็นน้ำเงินเข้ม** ทันที = รอยพับ (aliasing)
- (d) $Z_{DR}$ ในแกนฝนควรเป็นบวก (เม็ดใหญ่), (e) $\Phi_{DP}$ ที่เปลี่ยนสีวนรอบ ±180° = รอยพับเฟส, (f) $\rho_{HV}$ ต่ำที่ขอบพายุ/ระยะไกล/non-met ที่เหลือ

## 🌀 5. Doppler dilemma ของ KKN
ใช้ Nyquist ของ **volume ทั้งหมดที่ประมวลผล** (ไฟล์หลัก ± `NEIGHBOUR_WINDOW_MIN` นาที)
— ถ้าต้องการดูทั้งวันแบบต้นฉบับ ให้ตั้ง `NEIGHBOUR_WINDOW_MIN = 720`

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
vs = VOLSTATS[VOLSTATS["status"] == "ok"].copy()
c, lam = 2.998e8, pt.WAVELENGTH_M
tab = vs.groupby("nyquist").agg(n_volumes=("file", "size"), first=("time_utc", "min"), last=("time_utc", "max"))
tab["PRF_Hz (single PRF)"] = 4 * tab.index / lam
tab["r_max_km"] = c / (2 * tab["PRF_Hz (single PRF)"]) / 1000
tab.round(1)

In [ ]:
v = pt.field_data(radar, "velocity_qc")
vny = pt.nyquist(radar)
fig, axs = plt.subplots(1, 2, figsize=(8, 3), layout="constrained")
axs[0].plot(vs["time_utc"], vs["nyquist"], "o", ms=3, color="k")
axs[0].plot([T_TARGET], [vny], "o", ms=6, mfc="none", mec="r")
axs[0].set_ylabel("$V_{ny}$ (m s$^{-1}$)"); axs[0].set_ylim(0, 20)
axs[0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M")); axs[0].set_xlabel("Time (UTC)")
axs[0].set_title("(a) Nyquist velocity of processed volumes", loc="left")
axs[1].hist(v[np.isfinite(v)], bins=np.arange(-vny, vny + 0.5, 0.5), color="0.4")
for s_ in (-1, 1):
    axs[1].axvline(s_ * vny, color="r", ls="--", lw=1)
near = np.mean(np.abs(v[np.isfinite(v)]) > 0.85 * vny) * 100
axs[1].set_xlabel("$V_r$ (m s$^{-1}$)"); axs[1].set_ylabel("Number of gates")
axs[1].set_title(f"(b) $V_r$ at {T_TARGET:%H:%M} UTC: {near:.1f}% of gates beyond 0.85$V_{{ny}}$", loc="left")
mp.save_figure(fig, f"nb1_nyquist_{CASE_DATE}_{T_TARGET:%H%M}")
plt.show()
print(f"V_ny = {vny:.2f} m/s → PRF ≈ {4 * vny / lam:.0f} Hz, r_max ≈ {c / (2 * 4 * vny / lam) / 1000:.0f} km")

- ถ้า $V_{ny}$ ≈ **8.38 m/s** (เหมือนช่วงพายุของเคส 3 พ.ค.) ลมกระโชกของ downburst ที่มัก **> 20 m/s** จะพับแน่นอน
- ถ้า $V_{ny}$ ≈ 16.75 m/s แสดงว่าใช้ยุทธวิธีการสแกนอีกแบบ (ดูคำถามชวนคิดข้อ 1)
- (b) ถ้าไม่มีการพับ histogram ควรลดลงเรียบ ๆ ที่ขอบ ถ้าที่ ±$V_{ny}$ ยังมี gate จำนวนมาก = มีลมแรงกว่าที่วัดได้

## 📐 6. ลำคลื่นเรดาร์อยู่สูงแค่ไหน เทียบกับระดับเยือกแข็ง

In [ ]:
# จุดอ้างอิงที่แสดงบนกราฟ: จุดที่สนใจ + จุดรายงานหลัก (ถ้าเป็นเคส 3 พ.ค.)
SITES = {FOCUS_NAME.split(" (")[0]: FOCUS}
if reports is not None:
    for n_ in ("Phu Kradueng", "Na Klang", "Si Chiang Mai", "Bueng Kan"):
        r_ = kt.site(reports, n_)
        SITES[n_] = (r_["lon"], r_["lat"])
site_rng = {n_: float(np.hypot(*kt.lonlat_to_xy(meta, lo, la)) / 1000) for n_, (lo, la) in SITES.items()}

r_km = np.linspace(0, 240, 241)
alt0 = meta["radar_alt_m"]
fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")
cols = plt.cm.viridis(np.linspace(0, 0.85, radar.nsweeps))
for e, col in zip(radar.fixed_angle["data"], cols):
    hc = pt.beam_height_km(r_km, e, alt0)
    lo, hi = pt.beam_height_km(r_km, e - 0.5, alt0), pt.beam_height_km(r_km, e + 0.5, alt0)
    ax.fill_between(r_km, lo, hi, color=col, alpha=0.25, lw=0)
    ax.plot(r_km, hc, color=col, label=f"{e:.2f}°")
for (lbl, h), ls in zip(H.items(), ("-", "--", ":")):
    ax.axhline(h / 1000, color="tab:blue", ls=ls, lw=1)
    ax.text(2, h / 1000 + 0.1, lbl, color="tab:blue", fontsize=7)
for name, rr in site_rng.items():
    ax.axvline(rr, color="r" if name == FOCUS_NAME.split(" (")[0] else "0.5", lw=0.6)
    ax.text(rr, 13.6, name, rotation=90, fontsize=6.5, va="top", ha="right", color="0.3")
ax.set_xlim(0, 240); ax.set_ylim(0, 14)
ax.set_xlabel("Range (km)"); ax.set_ylabel("Height (km MSL)")
ax.legend(title="Elevation", loc="upper left", fontsize=7, title_fontsize=7)
ax.grid(lw=0.3, alpha=0.5)
mp.save_figure(fig, f"nb1_beam_height_{CASE_DATE}_{T_TARGET:%H%M}")
plt.show()
bt = pd.DataFrame({n_: [pt.beam_height_km(rr, e, alt0) for e in radar.fixed_angle["data"]] for n_, rr in site_rng.items()},
                  index=[f"{e:.2f}°" for e in radar.fixed_angle["data"]]).T
bt.insert(0, "range_km", list(site_rng.values()))
print("ความสูงกึ่งกลางลำคลื่น (km MSL) เหนือจุดอ้างอิง")
bt.round(1)

**คำถามสำหรับข้อมูลของคุณ:** ที่จุดที่สนใจ ลำคลื่นมุมต่ำสุดอยู่สูงกี่ km และมุมสูงสุดเข้าใกล้ระดับ 0 °C / −20 °C หรือยัง?
- ถ้าลำคลื่นต่ำสุด > 1–1.5 km → **เห็นลม outflow ใกล้พื้นได้จำกัด** (downburst เกิดในชั้น < 1 km)
- ถ้ามีมุมยกที่อยู่ทั้งใต้และเหนือระดับ 0 °C → เห็น **ทั้งชั้นฝนและชั้นน้ำแข็ง** เหมาะกับ HID (NB5)

## ✅ 7. ตรวจคุณภาพข้อมูลเบื้องต้น
### 7.1 qc_flag: แต่ละ gate ถูกตัดเพราะอะไร
ค่า `qc_flag` เป็นผลรวมของ bit — ใช้ `np.bitwise_and(flag, bit) > 0` เพื่อดูแต่ละสาเหตุ

In [ ]:
QC_BITS = {"no_echo/below floor": 1, "flare ray": 2, "non-met": 4, "speckle": 8,
           "σv invalid": 16, "V invalid": 32, "ΦDP texture high": 64}
flag = np.ma.filled(radar.fields["qc_flag"]["data"], 0).astype(int)
zr = pt.field_data(radar, "reflectivity_raw")
has_raw = np.isfinite(zr)
print(f"gate ที่มีค่า reflectivity_raw: {has_raw.sum():,}")
pd.Series({k: int(((flag & b) > 0)[has_raw].sum()) for k, b in QC_BITS.items()}, name="gates").to_frame()

### 7.2 ΦDP ดิบตามแนวรัศมี: รอยพับเฟส และทิศทางที่ "กลับด้าน"
เลือก ray ที่ผ่านจุดที่สนใจ (±15°) และมีแกนฝน ≥ 35 dBZ ยาวที่สุด

In [ ]:
sw_ = 1 if radar.nsweeps > 1 else 0
az_f, rng_f = pt.azimuth_to(meta, FOCUS[0], FOCUS[1])
z1 = pt.field_data(radar, "reflectivity_qc", sw_)
sl1 = radar.get_slice(sw_)
az1 = radar.azimuth["data"][sl1]
cand = np.where(np.abs(((az1 - az_f) + 180) % 360 - 180) < 15)[0]
ray = cand[np.argmax(np.nansum(z1[cand] > 35, axis=1))]          # ray ที่มีแกนฝนยาวที่สุดใกล้จุดที่สนใจ
rk = radar.range["data"] / 1000
fig, axs = plt.subplots(3, 1, figsize=(7, 5.2), sharex=True, layout="constrained")
axs[0].plot(rk, z1[ray], color="k", lw=1); axs[0].set_ylabel("$Z_H$ (dBZ)")
axs[1].plot(rk, pt.field_data(radar, "differential_phase_qc", sw_)[ray], ".", ms=2, color="tab:purple")
axs[1].set_ylabel("$\\Phi_{DP}$ raw (°)"); axs[1].set_ylim(-185, 185); axs[1].set_yticks([-180, -90, 0, 90, 180])
axs[2].plot(rk, pt.field_data(radar, "cross_correlation_ratio_qc", sw_)[ray], color="tab:green", lw=1)
axs[2].set_ylabel("$\\rho_{HV}$"); axs[2].set_ylim(0.6, 1.01); axs[2].set_xlabel("Range (km)")
for ax in axs:
    ax.grid(lw=0.3, alpha=0.5)
axs[0].set_title(f"(a–c) Ray at azimuth {az1[ray]:.1f}°, elevation {radar.fixed_angle['data'][sw_]:.2f}°, {TLAB}", loc="left")
mp.save_figure(fig, f"nb1_phidp_ray_{CASE_DATE}_{T_TARGET:%H%M}")
plt.show()
slopes = pt.phidp_slope_by_sweep(radar)
print("ความชันเฉลี่ยของ ΦDP (หลัง unwrap) ราย sweep [°/km, จำนวน ray]:", {k: (round(v_[0], 2), v_[1]) for k, v_ in slopes.items()})

- เมื่อผ่านแกนฝน $\Phi_{DP}$ ของเรดาร์ทั่วไป **ควรเพิ่มขึ้น** ตามระยะ — ในเคส 3 พ.ค. 2567 ข้อมูล KKN **ลดลง** (ความชันเป็นลบ) → ต้อง **กลับเครื่องหมาย** ก่อนคำนวณ $K_{DP}$
  **ตรวจกับข้อมูลของคุณ:** ความชันข้างบนเป็นลบเหมือนกันหรือไม่?
- ค่ากระโดดจาก −180° ไป +180° คือ **รอยพับเฟส (phase wrap)** ไม่ใช่สัญญาณจริง — ทั้งสองเรื่องแก้ใน **NB4**

---
## 🧪 แบบฝึกหัด

### 🧪 แบบฝึกหัด 1.1: PRF, ระยะไกลสุด และลมที่วัดได้
1. จาก $V_{ny}$ ของ **ไฟล์ของคุณ** และ λ ของ KKN จงคำนวณ PRF และ $r_{max}$
2. ถ้าลม downburst จริง = **+22 m/s** (ออกจากเรดาร์) เรดาร์จะวัดได้กี่ m/s? ($n$ เท่าไร?)
3. ถ้าอยากวัดลม ±30 m/s ได้โดยไม่พับ ด้วย PRF เดียว $r_{max}$ จะเหลือกี่ km?

<details>
<summary><b>💡 คำใบ้</b></summary>

- `PRF = 4 * vny / lam`, `rmax = c / (2 * PRF)`
- หา $n$ ที่ทำให้ $V_{obs} = V_{true} - 2nV_{ny}$ อยู่ในช่วง $[-V_{ny}, V_{ny}]$: `n = np.round(v_true / (2 * vny))`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 1.1 — เขียนโค้ดของคุณที่นี่

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 1.1
c, lam, vny = 2.998e8, pt.WAVELENGTH_M, pt.nyquist(radar)
prf = 4 * vny / lam
print(f"1) V_ny = {vny:.2f} m/s → PRF = {prf:.0f} Hz → r_max = {c / (2 * prf) / 1000:.0f} km "
      f"(ข้อมูลไปถึง {radar.range['data'][-1]/1000:.0f} km)")
v_true = 22.0
n = np.round(v_true / (2 * vny))
print(f"2) n = {n:.0f} → V_obs = {v_true - 2 * n * vny:+.2f} m/s")
prf30 = 4 * 30 / lam
print(f"3) V_ny = 30 m/s ต้องใช้ PRF = {prf30:.0f} Hz → r_max = {c / (2 * prf30) / 1000:.0f} km เท่านั้น")
print("   → Doppler dilemma: เรดาร์ปฏิบัติการจึงเลือกวัดไกล แล้วแก้การพับด้วยซอฟต์แวร์ (NB2)")

### 🧪 แบบฝึกหัด 1.2: ประมาณ ZDR bias จากฝนเบา (volume ของคุณ)
ในฝนเบา (Z 20–28 dBZ) เม็ดฝนเล็กเกือบกลม $Z_{DR}$ จริงควรอยู่ราว **0.2 dB** ถ้าค่าที่วัดได้ต่างจากนี้ = **bias** ของระบบ
ใช้ `pt.zdr_bias_light_rain(radar, snd)` กับทุกไฟล์ใน `VOLS` แล้วสรุป bias รายมุมยก (median)

**คำถาม:** ผลใกล้กับค่าที่งานวิจัย KKN ใช้แก้ (+0.20 dB ที่มุมต่ำสุด, +0.39 dB มุมอื่น) หรือไม่? ทำไมมุมสูงจึงมี gate น้อยกว่า?

<details>
<summary><b>💡 คำใบ้</b></summary>

- `out = pt.zdr_bias_light_rain(r, snd)` คืน dict `{sweep: (bias, n)}`
- เก็บลง list แล้ว `pd.DataFrame(rows).groupby("sweep").median()`
- ค่าที่ต้อง **บวกเพิ่ม** ให้ ZDR = −bias

</details>

In [ ]:
# ✍️ แบบฝึกหัด 1.2 — เขียนโค้ดของคุณที่นี่

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 1.2
rows = []
for f in VOLS:
    r = pt.read_volume(f)
    for s, (b, n) in pt.zdr_bias_light_rain(r, snd).items():
        rows.append({"time": pt.volume_time(r).strftime("%H:%M"), "sweep": s, "bias_dB": b, "n_gates": n})
zb = pd.DataFrame(rows)
summary = zb.groupby("sweep").agg(bias_median=("bias_dB", "median"),
                                  bias_iqr=("bias_dB", lambda x: x.quantile(.75) - x.quantile(.25)),
                                  gates_total=("n_gates", "sum"))
summary["correction_to_add"] = -summary["bias_median"]
summary["KKN_research_value"] = [pt.ZDR_CORR_BY_SWEEP.get(s, np.nan) for s in summary.index]
print(f"จาก {len(VOLS)} volume")
print(summary.round(2))
print("\n→ NaN = ฝนเบาไม่พอ (< 30 gate) ในมุมยกนั้น — เพิ่ม NEIGHBOUR_WINDOW_MIN เพื่อใช้หลาย volume")
print("  มุมสูงมี gate น้อย เพราะฝนเบาที่ต่ำกว่าระดับ 0 °C − 1 km อยู่ใกล้เรดาร์เท่านั้น")

### 🧪 แบบฝึกหัด 1.3: ρHV ลดลงตามระยะหรือไม่?
เลือก gate ที่เป็นฝนแน่ ๆ (Z ≥ 35 dBZ, ต่ำกว่าระดับ 0 °C) จากมุมยกต่ำสุดของทุก volume ใน `VOLS`
แล้วหา **median ของ $\rho_{HV}$ ทุกช่วงระยะ 20 km** พลอตกราฟ ρHV–ระยะ

**คำถาม:** ถ้า ρHV ลดลงตามระยะ จะมีผลอย่างไรกับการจำแนกหยาดน้ำฟ้า (NB5)? สาเหตุทางกายภาพคืออะไร?

<details>
<summary><b>💡 คำใบ้</b></summary>

- ระยะของ gate: `np.broadcast_to(r.range["data"] / 1000, (nrays, ngates))`
- ความสูง gate: `r.gate_altitude["data"]`
- `pd.cut(range_km, np.arange(0, 241, 20))` แล้ว `groupby(...).median()`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 1.3 — เขียนโค้ดของคุณที่นี่

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 1.3
recs = []
for f in VOLS:
    r = pt.read_volume(f)
    sl = r.get_slice(0)
    z = pt.field_data(r, "reflectivity_qc", 0)
    rho = pt.field_data(r, "cross_correlation_ratio_qc", 0)
    alt = r.gate_altitude["data"][sl]
    rr = np.broadcast_to(r.range["data"] / 1000, z.shape)
    m = (z >= 35) & np.isfinite(rho) & (alt < H["0 °C"] - 500)
    recs.append(pd.DataFrame({"range_km": rr[m], "rho": rho[m]}))
rh = pd.concat(recs)
if len(rh) < 50:
    print(f"⚠️ มี gate ฝนแรงเพียง {len(rh)} gate — อาจเป็นเวลาที่ไม่มีพายุ ลองเลือกไฟล์อื่นหรือเพิ่ม NEIGHBOUR_WINDOW_MIN")
rh["bin"] = pd.cut(rh["range_km"], np.arange(0, 241, 20))
g = rh.groupby("bin", observed=True)["rho"].agg(["median", "count", lambda x: x.quantile(0.25)])
g.columns = ["median", "count", "p25"]
mid = [b.mid for b in g.index]
fig, ax = plt.subplots(figsize=(5, 3), layout="constrained")
ax.plot(mid, g["median"], "o-", color="k", label="median")
ax.plot(mid, g["p25"], "s--", color="0.5", label="25th percentile")
for x_, n_ in zip(mid, g["count"]):
    ax.text(x_, 0.905, f"{n_}", fontsize=6, ha="center", color="0.4")
ax.set_xlabel("Range (km)"); ax.set_ylabel("$\\rho_{HV}$ in rain (Z ≥ 35 dBZ)"); ax.set_ylim(0.9, 1.0)
ax.legend(fontsize=7); ax.grid(lw=0.3, alpha=0.5)
mp.save_figure(fig, f"nb1_ex13_rhohv_range_{CASE_DATE}_{T_TARGET:%H%M}")
plt.show()
print(g.round(3))
print("\nสาเหตุที่ ρHV ลดลงตามระยะ: ลำคลื่นกว้างขึ้น (non-uniform beam filling), SNR ต่ำลง และลำคลื่นเข้าใกล้ชั้นละลาย")
print("ผลต่อ HID: ฝนที่ระยะไกลอาจถูกจัดเป็นคลาสที่มี ρHV ต่ำ (เช่น WS/MH) → ระวังเมื่อแปลผลที่ระยะ > 150 km")

---
## 📝 สรุป
- อ่านไฟล์ UF ดิบของ KKN จาก Drive → **gentle QC** → บันทึกเป็นไฟล์ polar รูปแบบเดียวกับ repo (`OUT_DIR/polar/`)
  ไฟล์เหล่านี้ใช้กับ NB2–NB6 ได้ โดยเปลี่ยน `pt.polar_files(DATA, ...)` เป็น `pt.polar_files(OUT_DIR, folder="polar")`
- ตรวจ **Nyquist**, จำนวนมุมยก และ **ความสูงลำคลื่นเหนือจุดที่สนใจ** ของวันของคุณ เพราะเป็นตัวกำหนดว่าวิเคราะห์อะไรได้
- ข้อมูล dual-pol ต้องเตรียมก่อนใช้: **ZDR มี bias**, **ΦDP พับเฟส/อาจกลับเครื่องหมาย**, **ρHV ลดลงตามระยะ**

### ❓ คำถามชวนคิด
1. ถ้า $V_{ny}$ ของวันของคุณ = 16.75 m/s แต่ข้อมูลยังไปถึง 240 km — เป็นไปได้อย่างไร? (คำใบ้: dual-PRF / staggered PRT) มีข้อเสียอะไร?
2. ที่จุดที่สนใจ เรดาร์ KKN มีข้อจำกัดอะไรในการศึกษา downburst? ข้อมูลอื่นที่ช่วยได้คืออะไร?
3. ถ้าไม่มี sounding ของวันนั้น (ใช้ค่าตั้งต้น) ผลของ ZDR bias และ HID จะคลาดเคลื่อนอย่างไร?

### 📚 เอกสารอ้างอิง
- Doviak, R. J., & Zrnić, D. S. (1993). *Doppler Radar and Weather Observations* (2nd ed.). Academic Press.
- Kumjian, M. R. (2013). Principles and applications of dual-polarization weather radar. Parts I–III. *J. Operational Meteor.*, 1, 226–274.
- Helmus, J. J., & Collis, S. M. (2016). The Python ARM Radar Toolkit (Py-ART). *J. Open Research Software*, 4, e25.
- ต้นฉบับ: nattaponm/Teach_HID_Doppler_KKN_summerStorm (NB00, NB01) — ข้อมูลเรดาร์: กรมอุตุนิยมวิทยา